### Imports & seeds

In [1]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from itertools import combinations

np.random.seed(42)
tf.random.set_seed(42)


### Load data

In [2]:
file_path = "data\\new_data\\Shanghai_T1DM_merged.xlsx"  
df = pd.read_excel(file_path)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Patient_ID", "Date"]).reset_index(drop=True)

df.head()


,Date,CGM (mg / dl),CBG (mg / dl),Blood Ketone (mmol / L),Dietary intake,饮食,Insulin dose - s.c.,Non-insulin hypoglycemic agents,"CSII - bolus insulin (Novolin R, IU)","CSII - basal insulin (Novolin R, IU / H)",Insulin dose - i.v.,Patient_ID
0,2021-07-30 16:43:00,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,NaN,1001_0_20210730
1,2021-07-30 16:58:00,124.2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,1001_0_20210730
2,2021-07-30 17:13:00,129.6,NaN,NaN,data not available,未记录,NaN,NaN,NaN,NaN,NaN,1001_0_20210730
3,2021-07-30 17:28:00,142.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001_0_20210730
4,2021-07-30 17:43:00,156.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001_0_20210730


### Clean numeric columns

In [3]:
def extract_numeric(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, (int, float)):
        return float(x)
    m = re.search(r"[\d\.]+", str(x))
    return float(m.group()) if m else 0.0


NUMERIC_COLS = [
    "CGM (mg / dl)",
    "CBG (mg / dl)",
    "Blood Ketone (mmol / L)",
    "Insulin dose - s.c.",
    "CSII - bolus insulin (Novolin R, IU)",
    "CSII - basal insulin (Novolin R, IU / H)",
    "Insulin dose - i.v."
]

for col in NUMERIC_COLS:
    df[col] = df[col].apply(extract_numeric)

df[NUMERIC_COLS].describe()


,CGM (mg / dl),CBG (mg / dl),Blood Ketone (mmol / L),Insulin dose - s.c.,"CSII - bolus insulin (Novolin R, IU)","CSII - basal insulin (Novolin R, IU / H)",Insulin dose - i.v.
count,4368.000000,4368.000000,4368.0,4368.000000,4368.000000,4368.000000,4368.0
mean,188.489835,5.848764,0.0,0.269460,0.035027,0.004579,0.0
std,74.547343,36.688940,0.0,1.538592,0.462379,0.047150,0.0
min,39.600000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
25%,127.800000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
50%,190.800000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
75%,243.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0
max,475.200000,396.000000,0.0,16.000000,10.000000,0.600000,0.0


### Feature list

In [4]:
TARGET = "CGM (mg / dl)"

ALL_FEATURES = [
    "CGM (mg / dl)",
    "CBG (mg / dl)",
    "Blood Ketone (mmol / L)",
    "Insulin dose - s.c.",
    "CSII - bolus insulin (Novolin R, IU)",
    "CSII - basal insulin (Novolin R, IU / H)",
    "Insulin dose - i.v."
]


### Sequence builder

In [5]:
def make_sequences(data, target_idx, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback, target_idx])
    return np.array(X), np.array(y)


### LSTM / GRU trainer

In [6]:
def train_rnn(
    df,
    features,
    model_type="LSTM",
    lookback=12,
    epochs=30,
    batch_size=32
):
    data = df[features].values

    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    target_idx = features.index(TARGET)

    X, y = make_sequences(data_scaled, target_idx, lookback)

    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = Sequential()
    if model_type == "LSTM":
        model.add(LSTM(64, input_shape=(lookback, X.shape[2])))
    else:
        model.add(GRU(64, input_shape=(lookback, X.shape[2])))

    model.add(Dense(1))
    model.compile(optimizer="adam", loss="mse")

    es = EarlyStopping(patience=5, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=0,
        shuffle=False
    )

    y_pred = model.predict(X_test, verbose=0)

    X_last = X_test[:, -1, :-1]

    y_pred_real = scaler.inverse_transform(
        np.hstack([X_last, y_pred])
    )[:, -1]

    y_test_real = scaler.inverse_transform(
        np.hstack([X_last, y_test.reshape(-1,1)])
    )[:, -1]

    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    return rmse


### Baseline run

In [7]:
baseline_lstm = train_rnn(df, ALL_FEATURES, "LSTM")
baseline_gru  = train_rnn(df, ALL_FEATURES, "GRU")

baseline_lstm, baseline_gru


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


(np.float64(0.01768610743219587), np.float64(0.015854177598070264))

### Feature extrication search

In [10]:
results = []

for r in range(len(ALL_FEATURES), 1, -1):
    for subset in combinations(ALL_FEATURES, r):

        if TARGET not in subset:
            continue

        rmse = train_rnn(df, list(subset), model_type="GRU")

        results.append({
            "features": subset,
            "rmse": rmse
        })

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df.head(10)

C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a

,features,rmse
0,"(CGM (mg / dl), CBG (mg / dl), Blood Ketone (m...",0.009326
1,"(CGM (mg / dl), CBG (mg / dl), Blood Ketone (m...",0.009571
2,"(CGM (mg / dl), Blood Ketone (mmol / L), Insul...",0.009601
3,"(CGM (mg / dl), CBG (mg / dl), Blood Ketone (m...",0.009626
4,"(CGM (mg / dl), CBG (mg / dl), Insulin dose - ...",0.009652
5,"(CGM (mg / dl), CBG (mg / dl), Blood Ketone (m...",0.009859
6,"(CGM (mg / dl), CBG (mg / dl), CSII - basal in...",0.009909
7,"(CGM (mg / dl), Blood Ketone (mmol / L), CSII ...",0.010007
8,"(CGM (mg / dl), Blood Ketone (mmol / L), Insul...",0.010184
9,"(CGM (mg / dl), Insulin dose - s.c., CSII - ba...",0.010195


### Lookback tuning

In [11]:
best_features = list(results_df.iloc[0]["features"])

for lb in [6, 12, 24, 36]:
    rmse = train_rnn(df, best_features, "GRU", lookback=lb)
    print(f"Lookback {lb}: RMSE = {rmse:.2f} mg/dL")


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Lookback 6: RMSE = 0.01 mg/dL


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Lookback 12: RMSE = 0.01 mg/dL


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Lookback 24: RMSE = 0.01 mg/dL


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Lookback 36: RMSE = 0.01 mg/dL


### Regularized model

In [12]:
def train_regularized(df, features, lookback=24):
    data = df[features].values
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    target_idx = features.index(TARGET)
    X, y = make_sequences(data_scaled, target_idx, lookback)

    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = Sequential([
        GRU(64, return_sequences=True),
        Dropout(0.3),
        GRU(32),
        Dense(1)
    ])

    model.compile(optimizer="adam", loss="mse")
    model.fit(X_train, y_train, epochs=40, batch_size=32, verbose=0)

    y_pred = model.predict(X_test, verbose=0)

    X_last = X_test[:, -1, :-1]
    y_pred_real = scaler.inverse_transform(
        np.hstack([X_last, y_pred])
    )[:, -1]

    y_test_real = scaler.inverse_transform(
        np.hstack([X_last, y_test.reshape(-1,1)])
    )[:, -1]

    return np.sqrt(mean_squared_error(y_test_real, y_pred_real))


### Final best score

In [13]:
final_rmse = train_regularized(df, best_features)
print("Final GRU RMSE:", final_rmse, "mg/dL")


Final GRU RMSE: 0.009725692238637179 mg/dL


Around 95–97%.

### HMM

In [14]:
from hmmlearn.hmm import GaussianHMM

states = pd.cut(
    df[TARGET],
    bins=[0, 70, 180, 500],
    labels=[0,1,2]
).astype(int)

hmm = GaussianHMM(n_components=3, covariance_type="diag", n_iter=100)
hmm.fit(states.values.reshape(-1,1))

pred_states = hmm.predict(states.values.reshape(-1,1))
accuracy = np.mean(pred_states == states.values)

accuracy


np.float64(0.21085164835164835)

Around 21.09%.